# Dashboard Generation Profiling Notebook

Parameterized (papermill) notebook regenerating the dashboard benchmark charts from an
artifact package produced by `scripts/run_dashboard_benchmark_suite.py`.

Charts:
1. **(a)** Baseline-config generation substage stacked columns + features/min (secondary axis), one chart per
   scenario, one bar per path (detached legacy, in-tree legacy, columnar_gpu).
2. **(b)** End-to-end throughput: baseline-config stacked columns of total run seconds (generation +
   conversion/JSON export for the legacy paths + DB import) with end-to-end features/min on the secondary axis.
3. **(c)** 2-way scaling view grouped by path (in-tree legacy vs columnar_gpu), one bar per swept config,
   with a separate features/min trend series per path group.
4. **(d)** 2-way GPU/CPU resource-dimension comparison (peak RSS / GPU / util).

In [ ]:
# Parameters (papermill)
package_path = "."

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

package_dir = Path(package_path)
data = json.loads((package_dir / "extracted_data.json").read_text(encoding="utf-8"))
manifest = json.loads((package_dir / "manifest.json").read_text(encoding="utf-8"))

SUMMARY_STAGES = data["summary_stages"]
PATH_DISPLAY = data["path_display"]
SCENARIO_DISPLAY = data["scenario_display"]
PATH_ORDER = ["detached_legacy", "current_legacy", "columnar_gpu"]
STAGE_COLORS = plt.get_cmap("tab10").colors

variants = data["variants"]
by_scenario = {}
for v in variants:
    by_scenario.setdefault(v["scenario"], []).append(v)
for group in by_scenario.values():
    group.sort(key=lambda v: (PATH_ORDER.index(v["path_key"]) if v["path_key"] in PATH_ORDER else 99, v["label"]))


def config_of(v):
    return (v["n_features_per_batch"], v["n_prompts_in_forward_pass"])


def baseline_config(group):
    """Baseline config = detached-legacy leg's config when present, else the smallest swept config."""
    detached = [v for v in group if v["path_key"] == "detached_legacy"]
    if detached:
        return config_of(detached[0])
    return min(config_of(v) for v in group)


def baseline_variants(group):
    base_cfg = baseline_config(group)
    ordered = []
    for path_key in PATH_ORDER:
        match = next((v for v in group if v["path_key"] == path_key and config_of(v) == base_cfg), None)
        if match is not None:
            ordered.append(match)
    return base_cfg, ordered


# Per-scenario showcase configs for charts (a)/(b): the largest feature-axis sweep shapes,
# where the legacy-vs-columnar_gpu contrast is most stark. Falls back to the baseline
# config when a package predates these sweep shapes or has fewer than two paths there.
SHOWCASE_CONFIGS = {"monology": (4096, 256), "rte": (2048, 128)}


def showcase_variants(scenario, group):
    preferred = SHOWCASE_CONFIGS.get(scenario)
    if preferred is not None:
        ordered = []
        for path_key in PATH_ORDER:
            match = next((v for v in group if v["path_key"] == path_key and config_of(v) == preferred), None)
            if match is not None:
                ordered.append(match)
        if len(ordered) >= 2:
            return preferred, ordered
    return baseline_variants(group)


# Split the n-prompt scaling sweep legs (they carry n_prompts_total) from the main benchmark
# legs so the path/config comparison charts are not polluted by sweep shapes.
by_scenario_main = {s: [v for v in g if not v.get("n_prompts_total")] for s, g in by_scenario.items()}
by_scenario_main = {s: g for s, g in by_scenario_main.items() if g}
by_scenario_sweep = {s: [v for v in g if v.get("n_prompts_total")] for s, g in by_scenario.items()}
by_scenario_sweep = {s: g for s, g in by_scenario_sweep.items() if g}

print("Package:", package_dir.resolve())
print("Source root:", manifest.get("source_root"))
print("Lineages:", json.dumps(manifest.get("lineages", {}), indent=2))
print("Scenarios:", {s: [f"{v['path_key']}@{config_of(v)}" for v in g] for s, g in by_scenario.items()})

## Chart (a): Baseline-config generation substage stacked columns + throughput

One bar per path at the accepted baseline config (RTE `512x128`, Monology `1024x256`); swept scaling
configs are shown in chart (c).

Shown at the per-scenario showcase configs (Monology 4096x256, RTE 2048x128) where the legacy/columnar_gpu contrast is most stark; falls back to the baseline config for packages without those shapes.

In [ ]:
def draw_stacked_bars(ax, xs, group, add_stage_labels=True):
    bottoms = np.zeros(len(group))
    for si, stage in enumerate(SUMMARY_STAGES):
        heights = np.array([v["substage_seconds"].get(stage) or 0.0 for v in group])
        ax.bar(
            xs,
            heights,
            0.55,
            bottom=bottoms,
            label=stage if add_stage_labels else None,
            color=STAGE_COLORS[si % len(STAGE_COLORS)],
        )
        bottoms += heights
    batch_totals = np.array([v["avg_batch_seconds"] or 0.0 for v in group])
    remainder = np.clip(batch_totals - bottoms, 0, None)
    ax.bar(
        xs,
        remainder,
        0.55,
        bottom=bottoms,
        label="other (batch_total remainder)" if add_stage_labels else None,
        color="#cccccc",
    )


def annotate_line(ax2, xs, values):
    finite = [val for val in values if val is not None]
    if finite:
        # keep ~15% headroom above the highest point so the offset value labels
        # (e.g. columnar_gpu features/min) stay inside the axes frame
        ax2.set_ylim(bottom=0, top=max(max(finite) * 1.15, ax2.get_ylim()[1]))
    for xi, val in zip(xs, values):
        if val is not None:
            ax2.annotate(f"{val:.0f}", (xi, val), textcoords="offset points", xytext=(0, 8), ha="center")


def merge_legends(fig, ax, ax2):
    handles1, labels1 = ax.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(handles1 + handles2, labels1 + labels2, loc="upper left", bbox_to_anchor=(1.08, 1.0), fontsize=8)
    fig.tight_layout()


for scenario, group in by_scenario_main.items():
    base_cfg, baseline_group = showcase_variants(scenario, group)
    if not baseline_group:
        continue
    x = np.arange(len(baseline_group))
    fig, ax = plt.subplots(figsize=(8, 6))
    draw_stacked_bars(ax, x, baseline_group)
    ax.set_ylabel("Steady-state seconds per batch")
    ax.set_xticks(x, [PATH_DISPLAY.get(v["path_key"], v["path_key"]) for v in baseline_group], rotation=10)
    cfg_label = f"{base_cfg[0]}x{base_cfg[1]}"
    ax.set_title(f"{SCENARIO_DISPLAY.get(scenario, scenario)}: substage timings by path ({cfg_label})")
    ax2 = ax.twinx()
    fpm = [v["throughput_features_per_min"] for v in baseline_group]
    ax2.plot(x, fpm, "ko--", markersize=9, label="features/min")
    annotate_line(ax2, x, fpm)
    ax2.set_ylabel("Features/min (generation)")
    ax2.set_ylim(bottom=0)
    merge_legends(fig, ax, ax2)
    plt.show()

## Chart (b): End-to-end throughput

Baseline-config total run wall seconds per path, stacked as generation (all batches), conversion/JSON export
(legacy paths only; import-side `jsonl.gz` conversion), and the remaining DB import time. Secondary axis is
**end-to-end** features/min (`features x batches x 60 / (generation_total + import_wall)`), the headline
reviewer throughput metric.

Shown at the per-scenario showcase configs (Monology 4096x256, RTE 2048x128); falls back to the baseline config for older packages.

In [ ]:
E2E_SEGMENTS = [
    ("generation", "Generation (all batches)", "tab:blue"),
    ("conversion", "Conversion/JSON export", "tab:orange"),
    ("db_import", "DB import (excl. conversion)", "tab:green"),
]


def e2e_segments(v):
    n_batches = len(v["completed_batches"]) or 1
    generation = (v["avg_batch_seconds"] or 0.0) * n_batches
    conversion = v["import_conversion_seconds"] or 0.0
    db_import = max((v["import_wall_seconds"] or 0.0) - conversion, 0.0)
    return {"generation": generation, "conversion": conversion, "db_import": db_import}


for scenario, group in by_scenario_main.items():
    base_cfg, baseline_group = showcase_variants(scenario, group)
    if not baseline_group:
        continue
    x = np.arange(len(baseline_group))
    fig, ax = plt.subplots(figsize=(8, 6))
    bottoms = np.zeros(len(baseline_group))
    for key, seg_label, color in E2E_SEGMENTS:
        heights = np.array([e2e_segments(v)[key] for v in baseline_group])
        ax.bar(x, heights, 0.55, bottom=bottoms, label=seg_label, color=color)
        bottoms += heights
    ax.set_ylabel("Total run wall seconds (generation + export + import)")
    ax.set_xticks(x, [PATH_DISPLAY.get(v["path_key"], v["path_key"]) for v in baseline_group], rotation=10)
    cfg_label = f"{base_cfg[0]}x{base_cfg[1]}"
    ax.set_title(f"{SCENARIO_DISPLAY.get(scenario, scenario)}: end-to-end throughput by path ({cfg_label})")
    ax2 = ax.twinx()
    e2e_fpm = [v["e2e_features_per_min"] for v in baseline_group]
    ax2.plot(x, e2e_fpm, "kD--", markersize=9, label="E2E features/min")
    annotate_line(ax2, x, e2e_fpm)
    ax2.set_ylabel("End-to-end features/min")
    ax2.set_ylim(bottom=0)
    merge_legends(fig, ax, ax2)
    plt.show()

## Chart (c): 2-way scaling view grouped by path

Bars are grouped by path (in-tree legacy, then columnar_gpu) with one bar per batch-shape config inside
each group. The features/min trend is drawn as a separate series per path group so each path's scaling
trend reads independently.

In [ ]:
TWOWAY = ["current_legacy", "columnar_gpu"]
FPM_STYLES = {"current_legacy": dict(color="black", marker="o"), "columnar_gpu": dict(color="tab:red", marker="s")}
GROUP_GAP = 1.0

for scenario, group in by_scenario_main.items():
    path_groups = []
    for path_key in TWOWAY:
        members = sorted((v for v in group if v["path_key"] == path_key), key=config_of)
        if members:
            path_groups.append((path_key, members))
    if not path_groups:
        continue
    fig, ax = plt.subplots(figsize=(max(8, 2.0 * sum(len(m) for _, m in path_groups)), 6))
    ax2 = ax.twinx()
    x_start = 0.0
    xticks, xlabels = [], []
    for gi, (path_key, members) in enumerate(path_groups):
        xs = x_start + np.arange(len(members), dtype=float)
        draw_stacked_bars(ax, xs, members, add_stage_labels=(gi == 0))
        fpm = [v["throughput_features_per_min"] for v in members]
        style = FPM_STYLES.get(path_key, {})
        ax2.plot(
            xs, fpm, linestyle="--", markersize=9, label=f"{PATH_DISPLAY.get(path_key, path_key)} features/min", **style
        )
        annotate_line(ax2, xs, fpm)
        xticks.extend(xs)
        xlabels.extend(f"{v['n_features_per_batch']}x{v['n_prompts_in_forward_pass']}" for v in members)
        ax.annotate(
            PATH_DISPLAY.get(path_key, path_key),
            ((xs[0] + xs[-1]) / 2, -0.12),
            xycoords=("data", "axes fraction"),
            ha="center",
            fontsize=10,
            fontweight="bold",
        )
        x_start = xs[-1] + 1.0 + GROUP_GAP
    ax.set_ylabel("Steady-state seconds per batch")
    ax.set_xticks(xticks, xlabels)
    ax.set_title(f"{SCENARIO_DISPLAY.get(scenario, scenario)}: in-tree legacy vs columnar_gpu by config")
    ax2.set_ylabel("Features/min (generation)")
    ax2.set_ylim(bottom=0)
    merge_legends(fig, ax, ax2)
    plt.show()

## Chart (d): 2-way GPU/CPU resource-dimension comparison

GiB metrics on the primary axis; GPU utilization on a secondary axis with a fixed 0-100% ceiling.

DB import timing is intentionally excluded here (see chart (b)); this chart focuses on host/GPU resource
utilization peaks.

In [ ]:
RESOURCE_GIB_METRICS = [
    ("max_tree_rss_gib", "Peak tree RSS (GiB)", 1.0),
    ("max_gpu_process_used_mib", "Peak GPU process (GiB)", 1.0 / 1024),
]
# Peak GPU utilization is omitted: it saturates at 100% for every path/config and is
# non-informative; the steady-state average is the discriminating signal.
GPU_UTIL_METRICS = [
    ("avg_gpu_util_percent_steady", "Avg GPU util (%, steady)"),
]
for scenario, group in by_scenario_main.items():
    two = sorted(
        (v for v in group if v["path_key"] in TWOWAY),
        key=lambda v: (TWOWAY.index(v["path_key"]), config_of(v)),
    )
    if not two:
        continue
    x = np.arange(len(RESOURCE_GIB_METRICS) + len(GPU_UTIL_METRICS))
    width = 0.8 / len(two)
    fig, ax = plt.subplots(figsize=(max(9, 1.4 * len(two) + 6), 5))
    ax2 = ax.twinx()
    for vi, v in enumerate(two):
        gib_vals = [(v.get(metric) or 0.0) * scale for metric, _, scale in RESOURCE_GIB_METRICS]
        cfg = f"{v['n_features_per_batch']}x{v['n_prompts_in_forward_pass']}"
        bars = ax.bar(
            x[: len(RESOURCE_GIB_METRICS)] + vi * width,
            gib_vals,
            width,
            label=f"{PATH_DISPLAY.get(v['path_key'])} ({cfg})",
        )
        # GPU utilization (avg steady-state + peak) lives on the secondary %-scaled axis
        for mi, (metric, _) in enumerate(GPU_UTIL_METRICS):
            ax2.bar(
                x[len(RESOURCE_GIB_METRICS) + mi] + vi * width,
                v.get(metric) or 0.0,
                width,
                color=bars[0].get_facecolor(),
            )
    ax.set_xticks(
        x + width * (len(two) - 1) / 2,
        [label for _, label, _ in RESOURCE_GIB_METRICS] + [label for _, label in GPU_UTIL_METRICS],
        rotation=10,
    )
    ax.set_ylabel("GiB")
    ax2.set_ylabel("GPU utilization (%)")
    ax2.set_ylim(0, 105)
    ax.set_title(f"{SCENARIO_DISPLAY.get(scenario, scenario)}: resource comparison (in-tree legacy vs columnar_gpu)")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.10, 1.0))
    fig.tight_layout()
    plt.show()

## Primary benchmark data

In [ ]:
import pandas as pd

rows = []
for scenario, group in by_scenario.items():
    for v in group:
        rows.append(
            {
                "scenario": SCENARIO_DISPLAY.get(scenario, scenario),
                "path": PATH_DISPLAY.get(v["path_key"], v["path_key"]),
                "config": f"{v['n_features_per_batch']}x{v['n_prompts_in_forward_pass']}"
                + (f"x{v['n_prompts_total']}" if v.get("n_prompts_total") else ""),
                "avg_batch_s": v["avg_batch_seconds"],
                "features_per_min": v["throughput_features_per_min"],
                "import_wall_s": v["import_wall_seconds"],
                "activation_rows": v["imported_activation_rows"],
                "e2e_features_per_min": v["e2e_features_per_min"],
            }
        )
pd.DataFrame(rows)

## Chart (e): prompt-dimension scaling — packaging substage timing vs n_prompts

Only rendered when the package contains prompt-dimension sweep legs (`--monology-sweep`, config
`features:prompts:nprompts`). Shows the two heaviest columnar packaging substages
(`feature_statistics_packaging`, `activation_histogram_packaging`) plus `batch_total` as a function of
total prompt count, for the in-tree legacy and columnar_gpu paths. The super-linear rise near the GPU
memory ceiling is the memory-pressure/allocator-thrash cliff.

**Reading throughput here vs the primary benchmark:** sweep legs may run under the opt-in
reduced-peak-memory flags (`--runner-columnar-max-staged-acts-bytes` / `--runner-columnar-row-chunk-size`);
when used, they are echoed in the chart footnote. The flags trade speed for peak GPU memory (host-side
activation staging + smaller packaging/selection chunks) while leaving outputs bit-identical, so sweep
features/min is expected to be substantially lower than the primary-benchmark features/min at the same
`features x fwd-minibatch` config — that gap is the cost of the memory mitigation, not a regression.

**Caveat:** this is a single-layer curve — the memory ceiling is layer-dependent (denser layers cross it
sooner), so run the sweep on a sparse AND a dense layer and read the two packages together.

In [ ]:
# Chart (e): prompt-axis packaging-substage timing with generation throughput.
# Primary axis: the two hot packaging substages vs n_prompts; secondary axis: features/min.
PROMPT_PATHS = ["current_legacy", "columnar_gpu"]
HOT_STAGES = ["feature_statistics_packaging", "activation_histogram_packaging"]
STAGE_STYLE = {
    "feature_statistics_packaging": dict(marker="o", linestyle="-"),
    "activation_histogram_packaging": dict(marker="s", linestyle="--"),
}
PATH_COLOR = {"current_legacy": "black", "columnar_gpu": "tab:red"}
_sweep_layer = manifest.get("prompt_sweep_layer", manifest.get("layer", "?"))
_mit_args = manifest.get("prompt_sweep_mitigation_args") or []

for scenario, sweep in by_scenario_sweep.items():
    fig, ax = plt.subplots(figsize=(9, 6))
    ax2 = ax.twinx()
    plotted = False
    for path_key in PROMPT_PATHS:
        members = sorted((v for v in sweep if v["path_key"] == path_key), key=lambda v: v["n_prompts_total"])
        if not members:
            continue
        xs = [v["n_prompts_total"] for v in members]
        for stage in HOT_STAGES:
            ys = [(v.get("substage_seconds") or {}).get(stage) for v in members]
            if all(y is None for y in ys):
                continue
            ax.plot(
                xs,
                ys,
                color=PATH_COLOR.get(path_key, "gray"),
                label=f"{PATH_DISPLAY.get(path_key, path_key)} · {stage}",
                **STAGE_STYLE.get(stage, {}),
            )
            plotted = True
        fpm = [v.get("throughput_features_per_min") for v in members]
        if any(t is not None for t in fpm):
            ax2.plot(
                xs,
                fpm,
                color=PATH_COLOR.get(path_key, "gray"),
                alpha=0.5,
                marker="^",
                linewidth=1.2,
                linestyle=":",
                label=f"{PATH_DISPLAY.get(path_key, path_key)} · features/min",
            )
            for x, t in zip(xs, fpm):
                if t is not None:
                    ax2.annotate(f"{t:.0f}", (x, t), textcoords="offset points", xytext=(0, 6), fontsize=8, ha="center")
    if not plotted:
        plt.close(fig)
        continue
    ax.set_xlabel("n_prompts (total)")
    ax.set_ylabel("Steady-state seconds per batch (packaging substage)")
    ax2.set_ylabel("Features/min (generation)")
    ax2.set_ylim(bottom=0)
    ax.set_xticks(sorted({v["n_prompts_total"] for v in sweep}))
    ax.set_title(
        f"{SCENARIO_DISPLAY.get(scenario, scenario)} · layer {_sweep_layer}: "
        "packaging substage timing + throughput vs n_prompts"
    )
    ax.grid(True, alpha=0.3)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    # legend outside the axes (as in charts a-d) so it cannot obscure data labels
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper left", bbox_to_anchor=(1.10, 1.0))
    _note = (
        "Single-layer curve (configurable via --prompt-sweep-layer) — the memory ceiling is set by the densest layer."
    )
    if _mit_args:
        _note += "\nSweep legs ran with opt-in reduced-peak-memory flags: " + " ".join(_mit_args)
    fig.text(0.5, -0.04, _note, ha="center", fontsize=8, style="italic")
    fig.tight_layout()
    plt.show()

## Chart (f): prompt-dimension scaling — peak GPU device memory vs n_prompts (the cliff driver)

The memory quantity that actually OOMs. Timing only spikes once the cliff is crossed, whereas peak GPU
memory shows the *approach* to the device memory ceiling as n_prompts grows — making the driver of chart
(e)'s super-linear timing explicit. The dashed line is the physical device ceiling (recorded in the
package manifest when available; assumed 24 GiB for older packages). Points near the line indicate the
config is close to its OOM boundary for the swept layer on the benchmark hardware; the opt-in
reduced-peak-memory flags noted in chart (e) lower the whole curve at the cost of throughput.

In [ ]:
# Chart (f): peak GPU device memory vs n_prompts.
_ceiling_gib = (manifest.get("gpu_device_total_mib") or 24 * 1024) / 1024.0
for scenario, sweep in by_scenario_sweep.items():
    fig, ax = plt.subplots(figsize=(9, 6))
    plotted = False
    for path_key in PROMPT_PATHS:
        members = sorted((v for v in sweep if v["path_key"] == path_key), key=lambda v: v["n_prompts_total"])
        if not members:
            continue
        xs = [v["n_prompts_total"] for v in members]
        ys = [(v.get("max_gpu_device_used_mib") or 0) / 1024.0 for v in members]
        if not any(ys):
            continue
        ax.plot(xs, ys, marker="D", color=PATH_COLOR.get(path_key, "gray"), label=PATH_DISPLAY.get(path_key, path_key))
        for x, y in zip(xs, ys):
            ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 6), fontsize=8, ha="center")
        plotted = True
    if not plotted:
        plt.close(fig)
        continue
    ax.axhline(
        _ceiling_gib,
        color="crimson",
        linestyle="--",
        linewidth=1,
        label=f"{_ceiling_gib:.0f} GiB device memory ceiling",
    )
    ax.set_xlabel("n_prompts (total)")
    ax.set_ylabel("Peak GPU device memory (GiB)")
    ax.set_xticks(sorted({v["n_prompts_total"] for v in sweep}))
    ax.set_ylim(0, _ceiling_gib * 1.09)
    ax.set_title(f"{SCENARIO_DISPLAY.get(scenario, scenario)} · layer {_sweep_layer}: peak GPU memory vs n_prompts")
    ax.grid(True, alpha=0.3)
    # legend outside the axes (as in charts a-d) so it cannot obscure the ceiling-line label or data labels
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.tight_layout()
    plt.show()